# Fundamentals 00.3 - Runtime OpenAI Provider API

Objetivo: probar la ruta `openai-runtime` de forma aislada antes de construir workflows mas grandes.

Este notebook usa el provider nativo de Agentic Systems. No usa `openai-agents`.

Regla de diseno:

```text
Agentic Systems define el contrato de ejecucion.
openai-runtime define el backend OpenAI directo.
Integrations no son obligatorias para usar OpenAI.
```


## 0) Imports minimos

El notebook asume que `agentic-systems` esta instalado en el ambiente activo.


In [ ]:
import json
import os

import agentic_systems as lab

print("agentic_systems:", lab.__name__)


## Problema default de fundamentals

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso.


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Problema default")


## 1) Utilidad segura de impresion


In [ ]:
def show_json(obj, title: str | None = None) -> None:
    if title:
        print(f"\n=== {title} ===")
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


## 2) RuntimeConfig y SchedulerConfig para OpenAI

Esta celda no llama a OpenAI. `lab.runtime(...)` lee la configuracion OpenAI del ambiente o `.env` igual que Bedrock lee su configuracion: modelo, base URL, proyecto y presencia de API key quedan visibles sin exponer secretos.


In [ ]:
scheduler = lab.scheduler(
    timeout_s=60,
    max_retries=1,
    max_tool_calls=5,
    max_turns=6,
    max_concurrency=1,
    backoff_s=0.2,
)

runtime = lab.runtime(
    provider="openai-runtime",
    region=None,
    scheduler=scheduler,
    metadata={"purpose": "fundamentals_openai_provider_notebook"},
)

show_json(runtime.describe(), "OpenAI runtime describe")


## 3) Configuracion OpenAI segura

Esta celda no pide ni guarda secretos. `openai-runtime` y `provider="auto"` leen `OPENAI_API_KEY` y la configuracion OpenAI desde el ambiente del kernel o `.env`.

Si `has_openai_api_key` es `False`, configura la variable fuera del notebook y reinicia/recarga el kernel.


In [ ]:
RUN_OPENAI_SMOKE_TESTS = bool(os.getenv("OPENAI_API_KEY"))

show_json(
    {
        "model": runtime.model_id,
        "openai_configuration": runtime.describe().get("configuration", {}).get("openai", {}),
        "run_openai_smoke_tests": RUN_OPENAI_SMOKE_TESTS,
        "has_openai_api_key": bool(os.getenv("OPENAI_API_KEY")),
        "auto_runtime_selection": lab.runtime(provider="auto", scheduler=scheduler).describe(),
    },
    "openai config",
)


## 4) Definir una tool local para el provider OpenAI

`openai-runtime` necesita tools concretas para ejecutar el loop nativo de tool calling. La tool sigue siendo una tool normal de Agentic Systems.


In [ ]:
@lab.tool
def sumar(a: int, b: int) -> dict:
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


agent = lab.agent(
    name="openai_runtime_smoke_agent",
    instructions="Usa la tool disponible y devuelve una respuesta breve con evidencia.",
    tools=[sumar],
    runtime=runtime,
    contract=lab.AgentContract(must_call=["sumar"], completion="when_required_tools_satisfied"),
    policy=lab.RunPolicy(max_tool_calls=1, max_turns=3, temperature=0.0),
)

lab.show(agent.info(), title="OpenAI runtime agent")


## 5) Smoke opcional con `openai-runtime`

Esta celda ejecuta el provider nativo de Agentic Systems cuando hay API key. Si no hay credenciales, el notebook sigue siendo ejecutable y explica que falta.


In [ ]:
if RUN_OPENAI_SMOKE_TESTS:
    result = agent.run("Suma 10 y 20 usando la tool sumar.", mode="eval")
    lab.human_result(result, pretty=False, show_lineage=True)
    lab.show(result.normalized(), title="OpenAI runtime normalized result")
else:
    print("Saltado: configura OPENAI_API_KEY en el ambiente del kernel para ejecutar openai-runtime.")


## 6) Variantes de `mode` para ejecuci?n evaluable

`mode` declara la intenci?n del run. No cambia la API p?blica del agente, pero s? resuelve una `RunPolicy` distinta y queda registrada en el `RunResult`.

En tutorials conviene escribirlo expl?citamente porque estamos ense?ando una ejecuci?n evaluable, no una llamada casual.


In [ ]:
run_modes = ["default", "fast", "eval", "audit", "debug", "prod"]

mode_variants = {
    mode: lab.RunPolicy.for_mode(mode).model_dump(mode="json")
    for mode in run_modes
}

lab.show(mode_variants, title="Run modes -> RunPolicy")


## 7) Patr?n recomendado

Para este notebook usamos `mode="eval"` porque queremos una corrida reproducible y validable. En producci?n cambia la intenci?n, no la fachada:

```python
agent.run(prompt, mode="eval")   # tutorial, smoke, evaluaci?n
agent.run(prompt, mode="prod")   # producci?n conservadora
agent.run(prompt, mode="debug")  # diagn?stico con traza m?s amplia
```

Si no escribes `mode`, Agentic Systems usa `default`.


## 8) Lectura correcta del diseno

- `openai-runtime` es provider/backend directo.
- No depende de `openai-agents`.
- La API publica sigue siendo `lab.runtime(...)`, `lab.agent(...)`, `lab.tool(...)` y `lab.human_result(...)`.
- `runtime.describe()` es la forma estable de auditar seleccion de backend.


## Coverage API de este notebook


In [ ]:
api_coverage = [
    {"api": "lab.runtime(provider='openai-runtime')", "description": "Declara OpenAI Runtime como backend canonico."},
    {"api": "RunPolicy.for_mode", "description": "Explica las variantes de mode sin ejecutar llamadas extra al provider."},
    {"api": "agent.run(..., mode='eval')", "description": "Declara una corrida evaluable y reproducible para tutorials."},
    {"api": "lab.scheduler", "description": "Declara limites de ejecucion antes de llamar al provider."},
    {"api": "RuntimeConfig.describe", "description": "Expone provider, modelo y scheduler sin ejecutar inferencia."},
    {"api": "lab.tool", "description": "Define una tool local compatible con tool calling."},
    {"api": "lab.agent", "description": "Crea un agente con runtime OpenAI nativo."},
    {"api": "lab.human_result", "description": "Renderiza el resultado cuando el smoke test esta activo."},
    {"api": "default problem declared", "description": "Mantiene el mismo problema de fundamentals para comparacion 1:1."},
]

lab.show({"notebook": "00_runtime_openai_provider_api.ipynb", "api_coverage": api_coverage})
